# Stretch: more about errors and tests, worked solutions

The answers to the stretch notebook. Try each one before looking. The
functions are in `solutions/stretch_solved.py` and the tests in
`solutions/test_stretch_solved.py`; the cells marked "run the tests" run
the matching part of that file.

In [ ]:
import importlib.util
import math
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

TESTS = "sessions/session-12/solutions/test_stretch_solved.py"


def run_pytest(*args):
    """Run pytest with these arguments and print what it says, if it is installed."""
    if importlib.util.find_spec("pytest") is None:
        print("pytest is not installed in this Python:  python -m pip install pytest")
        return None
    result = subprocess.run([sys.executable, "-m", "pytest", *args, "-p", "no:cacheprovider"],
                            capture_output=True, text=True)
    print(result.stdout)
    return result.returncode

## Stretch 1: `else` and `finally`

In [ ]:
def describe(text):
    """Print which parts of a try statement run for one piece of text."""
    try:
        number = float(text)
    except ValueError:
        print(f"  {text!r}: except  -> not a number")
    else:
        print(f"  {text!r}: else    -> got {number}")
    finally:
        print(f"  {text!r}: finally -> always")


describe("4.5")
describe("abc")

* `else` runs only when the `try` block raised nothing.
* `finally` runs every time, error or not.

**Why `else` and not the end of the `try` block?** To keep the `try` as
small as the one line that can fail. Code in `else` runs only on success,
but an error *inside* it is not caught by the `except` above, so it cannot
be mistaken for the error you meant to catch.

**What does the job of `finally` for files?** `with open(...)`. It closes
the file whether the block inside it finished or crashed, which is exactly
what a `finally: f.close()` would do. That is why you have needed `finally`
so rarely, and why you will not often write it yourself.

## Stretch 2: raise your own error

In [ ]:
def total(numbers):
    """Return the sum of a list of numbers."""
    running = 0
    for number in numbers:
        running += number
    return running


def average(numbers):
    """Return the mean of a list of numbers; ValueError for an empty list."""
    if len(numbers) == 0:
        raise ValueError("average() of an empty list: there is nothing to average")
    return total(numbers) / len(numbers)


try:
    average([])
except ValueError as error:
    print("ValueError:", error)

`ValueError` is the standard error for "the right type of thing, with a
value that makes no sense here". The message now says what the caller did
wrong, in words, which is the whole point of raising it yourself.

Notice that this is not `safe_average`. It still crashes, on purpose. It
just crashes with a message that saves the next person ten minutes.

The test checks the message too:

```python
def test_average_of_an_empty_list_says_why():
    with pytest.raises(ValueError, match="empty list"):
        average([])
```

Run the tests:

In [ ]:
assert run_pytest(TESTS, "-k", "average", "-v") in (0, None)

## Stretch 3: floats that are nearly equal

In [ ]:
def celsius_to_fahrenheit(c):
    """Return the Celsius temperature c converted to Fahrenheit."""
    return c * 9 / 5 + 32


print(celsius_to_fahrenheit(36.6))
print(celsius_to_fahrenheit(36.6) == 97.88)

print(average([0.1, 0.2]))
print(average([0.1, 0.2]) == 0.15)

print(math.isclose(celsius_to_fahrenheit(36.6), 97.88))
print(math.isclose(average([0.1, 0.2]), 0.15))

`0.1 + 0.2` is `0.30000000000000004`, so their average is a hair over
`0.15`. Neither function has a bug; asking floats for an *exact* match is
the bug. `math.isclose` asks "equal to within a tiny tolerance", and
`pytest.approx` is the same idea in test form:

```python
assert celsius_to_fahrenheit(36.6) == pytest.approx(97.88)
assert average([0.1, 0.2]) == pytest.approx(0.15)
```

Your session 3 homework dodged this with `round(..., 1)`, which also works
when you know how many decimals matter. Whole numbers are exact, so
`celsius_to_fahrenheit(0) == 32` needs neither.

In [ ]:
assert run_pytest(TESTS, "-k", "fahrenheit or tenths or freezing", "-v") in (0, None)

## Stretch 4: one test, many cases

```python
@pytest.mark.parametrize("text, expected", [
    ("4.65", 4.65),
    ("4.2 min", 4.2),
    (" 3.5 ", 3.5),
    ("0.32 min", 0.32),
    ("12", 12.0),
])
def test_to_minutes_reads_numbers(text, expected):
    assert to_minutes(text) == expected


@pytest.mark.parametrize("text", ["", "   ", "min", "about four", "4,2"])
def test_to_minutes_gives_none_for_non_numbers(text):
    assert to_minutes(text) is None
```

Two tests in the file, ten in the report: pytest runs each one once per row
and names each run after its data, so a failure tells you exactly which
case broke.

In [ ]:
assert run_pytest(TESTS, "-k", "minutes", "-v") in (0, None)

**The surprise:** `"4,2"`, with a decimal comma, gives `None`. `float()`
only understands a decimal point. For this dataset that is fine, but if
your own data comes from a spreadsheet set to Dutch or German, every number
in it might be written that way, and `pd.read_csv` has `decimal=","` for
exactly that. A parametrize table is a good place to write down the cases
you decided *not* to handle, so the decision is visible.

## Stretch 5: a test that needs a file

```python
def test_load_rows_reads_a_small_file(tmp_path):
    path = tmp_path / "plays.csv"
    path.write_text("device,minutes_played\nphone,4.65\ncar,3.10\n", encoding="utf-8")

    rows = load_rows(path)

    assert len(rows) == 2
    assert rows[0] == {"device": "phone", "minutes_played": "4.65"}   # still text


def test_load_rows_of_a_header_only_file_is_empty(tmp_path):
    path = tmp_path / "empty.csv"
    path.write_text("device,minutes_played\n", encoding="utf-8")
    assert load_rows(path) == []


def test_load_rows_missing_file_raises(tmp_path):
    with pytest.raises(FileNotFoundError):
        load_rows(tmp_path / "not_there.csv")
```

The second assert in the first test checks the session 4 lesson, that
everything from a CSV arrives as text: `"4.65"`, not `4.65`. A test is a
good place to pin down behaviour that surprised you once.

A header-only file gives an empty list, not an error, and that empty list
is exactly what makes `average` crash two steps later. Between this test
and Stretch 2, the whole chain is now written down: empty file, empty list,
a clear `ValueError`.

In [ ]:
assert run_pytest(TESTS, "-k", "load_rows", "-v") in (0, None)

## Stretch 6: pandas does not raise

In [ ]:
plays = pd.read_csv("data/clean/plays.csv")
radio = plays[plays["device"] == "radio"]
print(len(radio), "plays on the radio, mean of skipped:", radio["skipped"].mean())


def skip_rate(plays, device):
    """Return the percentage of plays on device that were skipped, or None."""
    on_device = plays[plays["device"] == device]
    if len(on_device) == 0:
        return None
    return on_device["skipped"].mean() * 100


print(f"car:   {skip_rate(plays, 'car'):.1f}%")
print(f"radio: {skip_rate(plays, 'radio')}")

assert round(skip_rate(plays, "car"), 1) == 10.6
assert skip_rate(plays, "radio") is None

The car is 19 skips out of 179 plays, the same 19 and 179 that came out as
`0` under integer division in session 6.

An `if` rather than a `try`, because there is no error to catch: pandas
returns `nan` and carries on. If you remember one thing from this stretch:
**pandas mostly does not crash on empty data, it hands you `NaN`**, so the
only thing standing between you and a chart with a hole in it is a check
you wrote on purpose, and a test that proves the check is there.

The tests use a fixture of their own, so three tests can share one tiny
DataFrame:

```python
@pytest.fixture
def plays():
    """Four plays you can check in your head: phone 1 of 2 skipped, car 0 of 2."""
    return pd.DataFrame({
        "device": ["phone", "phone", "car", "car"],
        "skipped": [1, 0, 0, 0],
    })


def test_skip_rate_on_the_phone(plays):
    assert skip_rate(plays, "phone") == 50


def test_skip_rate_with_no_plays_is_none(plays):
    assert skip_rate(plays, "radio") is None
```

Any test that names `plays` as an argument gets a fresh copy of that
DataFrame. That is exactly how `tmp_path` worked in Stretch 5; now you have
written a fixture yourself.

In [ ]:
assert run_pytest(TESTS, "-k", "skip or nan", "-v") in (0, None)

## All of them together

In [ ]:
assert run_pytest(TESTS) in (0, None)